In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt

from waffles.input_output.hdf5_structured import load_structured_waveformset
from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
# from waffles.data_classes.BasicWfAna import BasicWfAna
# from waffles.data_classes.IPDict import IPDict
from waffles.data_classes.UniqueChannel import UniqueChannel
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.utils.denoising.tv1ddenoise import Denoise
from waffles.np02_utils.AutoMap import generate_ChannelMap, dict_uniqch_to_module, dict_module_to_uniqch, ordered_modules_pmt, strUch, getModuleName
from waffles.np02_utils.PlotUtils import plot_detectors, runBasicWfAnaNP02, runBasicWfAnaNP02Updating, matplotlib_plot_WaveformSetGrid
from waffles.np02_utils.load_utils import open_processed

In [ ]:
dettype = 'pmt'
run=42862
nwaveforms=None
mergefiles=True
wfset_full = open_processed(run,dettype=dettype, datadir="/eos/experiment/neutplatform/protodune/experiments/ProtoDUNE-VD/commissioning/", nwaveforms=nwaveforms, mergefiles=mergefiles)


In [ ]:
diffvalues = [ wf.daq_window_timestamp - wf.timestamp for wf in wfset_full.waveforms  ]
dmax = np.max(diffvalues)
dmin = np.min(diffvalues)
print(dmax, dmin, (dmax-dmin)*16e-9)
hc, hbins, _ = plt.hist(diffvalues, bins=np.linspace(-50, 50, 100));
plt.xlabel("daq_window_timestamp - timestamp [ticks]")

In [ ]:
def adjust_offset(waveform:Waveform) -> bool:
    if -5 < waveform.daq_window_timestamp - waveform.timestamp < 25:
        return True
    return True
    
wfset_triggered_all = WaveformSet.from_filtered_WaveformSet(wfset_full, adjust_offset, show_progress=True)
wfset_triggered_all

In [ ]:
runBasicWfAnaNP02(wfset_triggered_all, baselinefinish=65, onlyoptimal=True, threshold=25)

In [ ]:
argsheat = dict(
    mode="heatmap",
    analysis_label="std",
    adc_range_above_baseline=1000,
    adc_range_below_baseline=-250,
    adc_bins=400,
    time_bins=wfset_full.points_per_wf//2,
    filtering=4,
    share_y_scale=True,
    share_x_scale=True,
    wfs_per_axes=5000,
    zlog=True,
    width=1600,
    height=1200
)
# detector=[ UniqueChannel(110, ch) for ch in wfch[110].keys() ]
detector = ordered_modules_pmt
group1 = detector[:len(detector)//2]
group2 = detector[len(detector)//2:]
plot_detectors(wfset_triggered_all, detector, **argsheat)

In [ ]:
def select_amp(waveform: Waveform) -> bool:
    slice_min = slice(75,80)
    ch = waveform.channel
    ep = waveform.endpoint
    if ch == 16:
        slice_min = slice(78,83)
    if np.min(waveform.adcs[slice_min]-waveform.analyses['std'].result['baseline']) > 400:
        if np.max(waveform.adcs[60:80]-waveform.analyses['std'].result['baseline']) < 7e3:
            return True
    return False
wfset = WaveformSet.from_filtered_WaveformSet(wfset_triggered_all, select_amp, show_progress=True)
wfch = ChannelWsGrid.clusterize_waveform_set(wfset)
wfch

In [ ]:
wftmp = wfset

plot_detectors(wfset, detector, **argsheat)


In [ ]:
from waffles.utils.numerical_utils import average_wf_ch
from waffles.utils.deconvolution.DeconvFitter import DeconvFitter
import math
def pmt_average_and_fit(wfset: WaveformSet):
    # denoiser = Denoise()
    deconv = DeconvFitter()
    average_wvf = {}
    ep = wfset.waveforms[0].endpoint
    ch = wfset.waveforms[0].channel

    wfm = average_wf_ch(wfset)
    wfm = wfm - np.mean(wfm[:50])
    deconv.deconvolved = wfm

    # oneexp = True if getModuleName(ep,ch) == "P35" else False
    oneexp=False
    deconv.fit(oneexp=oneexp, fit_limits_ns = [None, 10000])

    nticks = len(deconv.deconvolved)
    times = np.linspace(0, nticks*16, nticks, endpoint=False)

    plt.plot(times, deconv.deconvolved, '-', lw=2, color='k', label='data')

    if hasattr(deconv, 'fit_results') and len(deconv.fit_results) > 0:
        plt.plot(deconv.times, deconv.model(deconv.times, *deconv.fit_results), color='r', zorder=100, label='Fit Model')
        
        fit_info = [
            f"$\\chi^2$/$n_\\mathrm{{dof}}$ = {deconv.m.fval:.2f} / {deconv.m.ndof:.0f} = {deconv.m.fmin.reduced_chi2:.2f}",
        ]

        mapnames = {
            'A': 'Norm.',
            'fp': 'A_\\text{fast}',
            'fs': 'A_\\text{slow}',
            't1': '\\tau_\\text{fast}',
            't3': '\\tau_\\text{slow}',
            'td': '\\tau_\\text{delay}',
            't0': 't_\\text{0}',
            'sigma': '\\sigma',
        }

        for p, v, e in zip(deconv.m.parameters, deconv.m.values, deconv.m.errors):
            if p in mapnames:
                fit_info.append(f"${mapnames[p]}$ = ${v:.3f} \\pm {e:.3f}$")
        plt.plot([], [], ' ', label='\n'.join(fit_info))

    plt.title(f"{getModuleName(ep,ch)}: {ep}-{ch}")
    plt.yscale('log')
    plt.legend(fontsize=8)
    plt.ylim(1e-2, np.max(wfm)*1.5)
    plt.ylabel('Amplitude [ADC]')
    plt.xlabel('Time [ns]')
    plt.xlim(0, 12000)
    # plt.savefig('out.png')

In [ ]:
fig, axs = matplotlib_plot_WaveformSetGrid(wftmp, detector=group1, plot_function=pmt_average_and_fit, figsize=(20,20))
plt.tight_layout()
plt.savefig('pmts1.png')
plt.show()

fig, axs = matplotlib_plot_WaveformSetGrid(wftmp, detector=group2, plot_function=pmt_average_and_fit, figsize=(20,20))
plt.tight_layout()
plt.savefig('pmts2.png')
plt.show()

In [ ]:
for ch, wf in average_wvf.items():
    print(ch, np.shape(wf))

In [ ]:
import matplotlib.pyplot as plt

it+=1
plt.plot(wfset_triggered_all.waveforms[it].adcs - wfset_triggered_all.waveforms[it].analyses['std'].result['baseline'])
plt.xlabel('ticks')
plt.ylabel('adcs')
plt.ylim(None, 50)
plt.xlim(0,75)